# Parse the test cases and descriptions

In [20]:
import json
import types

import bs4
import html2text

## Parse question 70

In [2]:
question = json.load(open("leet0070.json"))["data"]["question"]
question.keys()

dict_keys(['questionId', 'questionFrontendId', 'boundTopicId', 'title', 'titleSlug', 'content', 'translatedTitle', 'translatedContent', 'isPaidOnly', 'difficulty', 'likes', 'dislikes', 'isLiked', 'similarQuestions', 'contributors', 'langToValidPlayground', 'topicTags', 'companyTagStats', 'codeSnippets', 'stats', 'hints', 'solution', 'status', 'sampleTestCase', 'metaData', 'judgerAvailable', 'judgeType', 'mysqlSchemas', 'enableRunCode', 'enableTestMode', 'envInfo', 'libraryUrl', '__typename'])

In [3]:
# Parse content
soup = bs4.BeautifulSoup(question["content"], "html.parser")
content_as_text = soup.text
print(content_as_text)

You are climbing a staircase. It takes n steps to reach the top.
Each time you can either climb 1 or 2 steps. In how many distinct ways can you climb to the top?
 
Example 1:

Input: n = 2
Output: 2
Explanation: There are two ways to climb to the top.
1. 1 step + 1 step
2. 2 steps

Example 2:

Input: n = 3
Output: 3
Explanation: There are three ways to climb to the top.
1. 1 step + 1 step + 1 step
2. 1 step + 2 steps
3. 2 steps + 1 step

 
Constraints:

1 <= n <= 45




In [7]:
# Create the description for README.md by converting HTML to markdown
description = html2text.html2text(question["content"])
print(description)

You are climbing a staircase. It takes `n` steps to reach the top.

Each time you can either climb `1` or `2` steps. In how many distinct ways can
you climb to the top?



**Example 1:**

    
    
    **Input:** n = 2
    **Output:** 2
    **Explanation:** There are two ways to climb to the top.
    1. 1 step + 1 step
    2. 2 steps
    

**Example 2:**

    
    
    **Input:** n = 3
    **Output:** 3
    **Explanation:** There are three ways to climb to the top.
    1. 1 step + 1 step + 1 step
    2. 1 step + 2 steps
    3. 2 steps + 1 step
    



**Constraints:**

  * `1 <= n <= 45`




In [12]:
# Parse the examples
out = []
for line in content_as_text.splitlines():
    if line.startswith("Example "):
        test_id = line.strip().removesuffix(":")
        out.append(types.SimpleNamespace(test_id=test_id))
out

[namespace(test_id='Example 1'), namespace(test_id='Example 2')]

In [18]:
# Parse output
# Output: 3
# Output: [1,3,3,1]


def parse_output(text: str):
    assert text.startswith("Output: ")
    text = text.removeprefix("Output: ")
    text = text.strip()
    text = text.replace("'", '"')
    output = json.loads(text)
    return output


samples = [
    "Output: [1,3,3,1]\n",
    "Output: ['one', 'two']",
    "Output: 3\n",
    "Output: true\n",
]

for sample in samples:
    output = parse_output(sample)
    print(f"{sample!r} -> {output!r}")

'Output: [1,3,3,1]\n' -> [1, 3, 3, 1]
"Output: ['one', 'two']" -> ['one', 'two']
'Output: 3\n' -> 3
'Output: true\n' -> True


## Parse Input

This is harder. Here are some samples:

```
Input: n = 19
Input: head = [1,2,6,3,4,5,6], val = 6
Input: head = [], val = 1
Input: head = [7,7,7,7], val = 7
Input: s = "egg", t = "add"
Input: s = "f11", t = "b23"
Input: s = "paper", t = "title"
Input: nums = [1,2,3,1]
Input: nums = [1,2,3,1], k = 3
```

In [36]:
text = "Input: head = [1,2,6,3,4,5,6], val = 6\n"
text = text.strip().removeprefix(
    "Input: ",
)
for token in text.split(", "):
    print(repr(token))
    name, value = token.split(" = ")
    value = json.loads(value)
    print(f"{name=}, {value=}")

'head = [1,2,6,3,4,5,6]'
name='head', value=[1, 2, 6, 3, 4, 5, 6]
'val = 6'
name='val', value=6


In [32]:
"Input: n = 5\n".split(", ")

['Input: n = 5\n']

In [49]:
def parse_single_line_input(text: str):
    if not text.startswith("Input: "):
        return False, None

    text = text.removeprefix(
        "Input: ",
    ).strip()
    tokens = text.split(", ")

    name_value = {}
    for token in tokens:
        name, value = token.split(" = ")
        value = json.loads(value)
        name_value[name] = value

    return True, name_value


def parse_input(text: str):
    parsers = [parse_single_line_input]
    for parser in parsers:
        ok, parsed = parser(text)
        if ok:
            return ok, parsed
    return False, f"Cannot parse: {text!r}"

In [58]:
samples = """Input: n = 19
Input: head = [1,2,6,3,4,5,6], val = 6
Input: head = [], val = 1
Input: head = [7,7,7,7], val = 7
Input: s = "egg", t = "add"
Input: s = "f11", t = "b23"
Input: s = "paper", t = "title"
Input: nums = [1,2,3,1]
Input: nums = [1,2,3,1], k = 3
foo bar
Input
""".splitlines()

for sample in samples:
    sample = sample + "\n"
    try:
        ok, out = parse_input(sample)
        print(f"{sample!r:<50} -> {ok=}, {out!r}")
    except ValueError as error:
        print(error)

'Input: n = 19\n'                                  -> ok=True, {'n': 19}
'Input: head = [1,2,6,3,4,5,6], val = 6\n'         -> ok=True, {'head': [1, 2, 6, 3, 4, 5, 6], 'val': 6}
'Input: head = [], val = 1\n'                      -> ok=True, {'head': [], 'val': 1}
'Input: head = [7,7,7,7], val = 7\n'               -> ok=True, {'head': [7, 7, 7, 7], 'val': 7}
'Input: s = "egg", t = "add"\n'                    -> ok=True, {'s': 'egg', 't': 'add'}
'Input: s = "f11", t = "b23"\n'                    -> ok=True, {'s': 'f11', 't': 'b23'}
'Input: s = "paper", t = "title"\n'                -> ok=True, {'s': 'paper', 't': 'title'}
'Input: nums = [1,2,3,1]\n'                        -> ok=True, {'nums': [1, 2, 3, 1]}
'Input: nums = [1,2,3,1], k = 3\n'                 -> ok=True, {'nums': [1, 2, 3, 1], 'k': 3}
'foo bar\n'                                        -> ok=False, "Cannot parse: 'foo bar\\n'"
'Input\n'                                          -> ok=False, "Cannot parse: 'Input\\n'"
